<a href="https://colab.research.google.com/github/Animesh-Maiti/Named-Entity-Recognition-for-Drug-and-Symptom-Extraction/blob/main/application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### BioBERT Medical Entity Extraction using Gradio

This notebook demonstrates an application that uses a pre-trained BioBERT model to extract medical entities (DRUG, SYMPTOM) from PDF documents. It utilizes Hugging Face's `transformers` library for the NLP pipeline and `Gradio` for building an interactive web interface.

In [ ]:
# Install necessary libraries
!pip install PyPDF2 transformers gradio

### 0. Import Libraries and Mount Google Drive

This section imports all required Python libraries and mounts Google Drive to access the BioBERT model files.

In [ ]:
import gradio as gr
from PyPDF2 import PdfReader
import torch
from transformers import pipeline
import pandas as pd
import os
from google.colab import drive

# --- STEP 0: MOUNT DRIVE ---
# This ensures we can see the /content/drive folder
drive.mount('/content/drive', force_remount=True)

### 1. Setup the BioBERT NER Pipeline

Here, the BioBERT model is loaded from Google Drive and initialized as a Named Entity Recognition (NER) pipeline using Hugging Face's `transformers`. The `device` is set to use a GPU if available for faster inference.

In [ ]:
# --- STEP 1: SETUP THE PIPELINE ---
# Use absolute path to prevent Hugging Face Hub "Repo ID" errors
raw_path = "/content/drive/MyDrive/biobert-ner-final"
model_path = os.path.abspath(raw_path)

if not os.path.exists(model_path):
    print(f"❌ ERROR: Folder not found at {model_path}")
    print("Check if 'biobert-ner-final' is directly in your MyDrive folder.")
else:
    print(f"✅ Model found! Loading from: {model_path}")
    # Initialize the pipeline
    # device=0 uses the Colab T4 GPU, -1 uses CPU
    nlp = pipeline("ner",
                   model=model_path,
                   tokenizer=model_path,
                   aggregation_strategy="simple",
                   device=0 if torch.cuda.is_available() else -1)

### 2. Helper Functions

This section defines a utility function `map_label` to convert raw model output labels into more descriptive categories (e.g., 'LABEL_1' to 'DRUG').

In [ ]:
# --- STEP 2: HELPER FUNCTIONS ---
def map_label(label_raw):
    label_map = {
        "LABEL_0": "O",
        "LABEL_1": "DRUG",
        "LABEL_2": "DRUG",
        "LABEL_3": "SYMPTOM",
        "LABEL_4": "SYMPTOM"
    }
    return label_map.get(label_raw, label_raw)

### 3. Main Processing Logic (`process_document` function)

This core function handles the entire workflow:
1.  **Extract Text from PDF**: Reads the uploaded PDF file and extracts all text content.
2.  **Run Inference**: Passes the extracted text through the BioBERT NER pipeline.
3.  **Advanced Merging Logic**: Merges fragmented entities (e.g., "Am" + "oxicillin" into "Amoxicillin") to form complete entities.
4.  **Format for UI**: Prepares the extracted entities for display in the Gradio interface, including highlighting and tabular data.
5.  **Export CSV**: Generates a CSV file containing the extracted entities, categories, and confidence scores.

In [ ]:
# --- STEP 3: MAIN PROCESSING LOGIC ---
def process_document(file):
    if file is None:
        return "No file uploaded", [], [], None

    # Step A: Extract Text from PDF
    try:
        reader = PdfReader(file.name)
        full_text = ""
        for page in reader.pages:
            text = page.extract_text()
            if text:
                full_text += text + "\n"
    except Exception as e:
        return f"Error reading PDF: {str(e)}", [], [], None

    # Step B: Run Inference
    results = nlp(full_text)

    # Step C: Advanced Merging Logic (Fixes 'Am' + 'oxicillin')
    merged_results = []
    if results:
        current_ent = results[0].copy()
        current_ent['entity_group'] = map_label(current_ent['entity_group'])

        for next_ent in results[1:]:
            next_label = map_label(next_ent['entity_group'])

            # Merge if same label AND they are adjacent
            if next_label == current_ent['entity_group'] and next_ent['start'] <= current_ent['end'] + 2:
                current_ent['end'] = next_ent['end']
                current_ent['score'] = max(current_ent['score'], next_ent['score'])
            else:
                if current_ent['entity_group'] != "O":
                    merged_results.append(current_ent)
                current_ent = next_ent.copy()
                current_ent['entity_group'] = next_label

        if current_ent['entity_group'] != "O":
            merged_results.append(current_ent)

    # Step D: Format for UI
    highlights = []
    table_data = []
    last_idx = 0

    for entity in merged_results:
        if entity['start'] > last_idx:
            highlights.append((full_text[last_idx:entity['start']], None))

        entity_text = full_text[entity['start']:entity['end']].strip()
        label = entity['entity_group']

        highlights.append((entity_text, label))
        table_data.append([entity_text, label, f"{entity['score']:.2%}"])
        last_idx = entity['end']

    if last_idx < len(full_text):
        highlights.append((full_text[last_idx:], None))

    # Step E: Export CSV
    csv_path = "extracted_medical_entities.csv"
    if table_data:
        df = pd.DataFrame(table_data, columns=["Entity Name", "Category", "Confidence"])
        df.to_csv(csv_path, index=False)
    else:
        csv_path = None

    return full_text, highlights, table_data, csv_path

### 4. Gradio UI Design

This section constructs the user interface using `Gradio`. It includes components for uploading PDFs, displaying highlighted text with detected entities, showing raw extracted text, presenting a summary table of entities, and providing a download link for the CSV report.

In [ ]:
# --- STEP 4: GRADIO UI DESIGN ---
with gr.Blocks(theme=gr.themes.Soft(), title="BioBERT NER System") as demo:
    gr.Markdown("""
    # 🏥 BioBERT Medical Entity Extractor
    **B.Tech CSE (AI/ML) Mini-Project** | Automatic Drug & Symptom Extraction
    """)

    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(label="Upload Medical Report (PDF)", file_types=[".pdf"])
            with gr.Row():
                clear_btn = gr.Button("Clear")
                submit_btn = gr.Button("Extract Entities", variant="primary")

            gr.Markdown("### 📥 Export Results")
            download_file = gr.File(label="Download CSV Report")

        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.TabItem("✨ Highlighted Analysis"):
                    output_highlight = gr.HighlightedText(
                        label="NER Visualizer",
                        combine_adjacent=True,
                        color_map={"DRUG": "#ffd700", "SYMPTOM": "#ff6b6b"}
                    )
                with gr.TabItem("📝 Raw Text Preview"):
                    output_text = gr.Textbox(label="Extracted Text", lines=15, interactive=False)

    gr.Markdown("### 📋 Entity Summary Table")
    output_table = gr.Dataframe(
        headers=["Entity Name", "Category", "Confidence"],
        datatype=["str", "str", "str"],
        label="Detected Entities"
    )

    submit_btn.click(
        fn=process_document,
        inputs=file_input,
        outputs=[output_text, output_highlight, output_table, download_file]
    )

    clear_btn.click(lambda: (None, "", [], [], None), None, [file_input, output_text, output_highlight, output_table, download_file])

### 5. Launch the Gradio Application

This cell launches the Gradio interface, making it accessible via a public share link. The `debug=True` option provides additional logging in the console.

In [ ]:
# Launching with a public share link
demo.launch(debug=True, share=True)